### Extrator - Itaú

In [1]:
import pdfplumber
import pandas as pd
import re
import numpy as np
import json

In [2]:
# para PE e exposição bruta:

# exposição b.: b) Valor Contábil Bruto por Estágios ( pág 52 )
# PE: "c) Perda de Crédito Esperada ( pág 54-55)


In [3]:
with pdfplumber.open("Demonstrações Financeiras Itaú - 4T25.pdf") as pdf:
    page = pdf.pages[61]   # página 53 (índice começa em 0)
    text = page.extract_text()

# Remove espaços em branco extras de cada linha ( pré-processamento )
text = "\n".join(line.strip() for line in text.splitlines())

# Define início ( start ) e fim ( end ) do trecho a ser extraído. 
start = text.find("b) Valor Contábil Bruto por Estágios")
end = text.find("Consolidado dos 3 Estágios", start)

trecho = text[start:end]
print(trecho)

# divide em linhas para facilitar a criação do dataframe
linhas = [l.strip() for l in trecho.splitlines() if l.strip()]

b) Valor Contábil Bruto por Estágios
01/01/2025
Saldoem Transferência Transferênciapara Transferência Transferência Aquisição/ Saldoem
Estágio1 31/12/2024 paraEstágio2 Estágio3(1) doEstágio2 doEstágio3 (Liquidação) WriteOff 31/12/2025
PessoasFísicas 347.749 (29.288) (4.101) 36.920 355 59.172 - 410.807
PessoasJurídicas 332.440 (8.619) (2.135) 6.727 506 30.346 - 359.265
UnidadesExternas
196.464 (10.101) (1.166) 9.542 1.347 14.859 - 210.945
AméricaLatina
Total 876.653 (48.008) (7.402) 53.189 2.208 104.377 - 981.017
Saldoem Transferência Transferênciapara Transferência Transferência Aquisição/ Saldoem
Estágio2 WriteOff
31/12/2024 paraEstágio1 Estágio3 doEstágio1 doEstágio3 (Liquidação) 31/12/2025
PessoasFísicas 66.468 (36.920) (14.712) 29.288 6.652 (15.907) - 34.869
PessoasJurídicas 13.237 (6.727) (6.220) 8.619 2.176 (1.339) - 9.746
UnidadesExternas
14.004 (9.542) (4.474) 10.101 2.287 (2.047) - 10.329
AméricaLatina
Total 93.709 (53.189) (25.406) 48.008 11.115 (19.293) - 54.944
Saldoem Tran

In [4]:
# basicamente, estou sempre criando uma lista vazia nos ifs, e decidindo o que colocar nesta lista vazia ( que seria
# a saída ). Eu posso definir regras do que deve entrar e o que deve sair desta lista

PRODUTOS = [
    "PessoasFísicas",
    "PessoasJurídicas"
]

# verifica se a linha inicia com o nome de um produto
def normalizar(txt: str) -> str:
    return txt.replace(" ", "")

def eh_produto(linha: str) -> bool:
    linha_n = normalizar(linha)
    return any(linha_n.startswith(normalizar(p)) for p in PRODUTOS)

# pega apenas o primeiro e ó último elemento de uma linha
def reduzir_linha_produto(linha: str) -> str:
    partes = linha.split()
# ao invés de apenas o último elemento, pegue o último e o penúltimo
    ultimo = partes[-1]

    for produto in PRODUTOS:
        # compara sem espaços
        if normalizar(linha).startswith(normalizar(produto)):
            qtd_palavras_produto = len(produto.split())

            primeiro_valor = partes[qtd_palavras_produto]

            return f"{produto.replace(' ', '')} {primeiro_valor} {ultimo}"

    return linha


# cria um novo dataframe baseado no que foi feito
def reduzir_texto(texto: str) -> str:
    linhas = [l.strip() for l in texto.splitlines() if l.strip()]
    saida = []
    ignorando_bloco = False

    for linha in linhas:


        # Enquanto estiver dentro do bloco, ignore tudo
        if linha.startswith("UnidadesExternas"):
            ignorando_bloco = True
            continue

        # Enquanto estiver dentro do bloco, ignore tudo
        if ignorando_bloco:
            # FIM do bloco
            if linha.startswith("AméricaLatina"):
                ignorando_bloco = False
            continue

        # Regra — remover Total
        if linha.startswith("Total"):
            continue

        # Regra — remover Saldo
        if linha.startswith("Saldo"):
            continue
        
        # Regra — remover Consolidado e o resto
        if linha.startswith("Consolidado"):
            break
        
        # Isso é para datas
        if linha[:10].count("/") == 2:
            continue

        
        # para conservar apenas as linhas de estágio
        
        # Estágio 1
        if linha.startswith("Estágio1"):
            saida.append("Estágio1")
            continue

        # Estágio 2
        if linha.startswith("Estágio2"):
            saida.append("Estágio2")
            continue

        # Estágio 3
        if linha.startswith("Estágio3"):
            saida.append("Estágio3")
            continue

        # Redução dos produtos
        if eh_produto(linha):
            saida.append(reduzir_linha_produto(linha))
        else:
            saida.append(linha)

    return "\n".join(saida)



In [5]:
texto_reduzido = reduzir_texto(trecho)
print(texto_reduzido)

b) Valor Contábil Bruto por Estágios
Estágio1
PessoasFísicas 347.749 410.807
PessoasJurídicas 332.440 359.265
Estágio2
PessoasFísicas 66.468 34.869
PessoasJurídicas 13.237 9.746
Estágio3
PessoasFísicas 31.357 27.550
PessoasJurídicas 11.956 11.277


In [6]:
# para criar a coluna de trimestre

from datetime import datetime

padrao_data = re.compile(r"\b\d{2}/\d{2}/\d{4}\b")
datas = padrao_data.findall(trecho)

print(datas)

def trimestre_from_date(date_str: str) -> str:
    dt = datetime.strptime(date_str, "%d/%m/%Y")
    trimestre_map = {3: "1T", 6: "2T", 9: "3T", 12: "4T"}
    trimestre = trimestre_map.get(dt.month)

    if not trimestre:
        raise ValueError(f"Mês inesperado na data: {date_str}")

    return f"{trimestre}{str(dt.year)[-2:]}"

def extract_anos(texto: str) -> list[str]:
    datas = re.findall(r"\d{2}/\d{2}/\d{4}", texto)

    if len(datas) < 2:
        raise ValueError("Não foi possível encontrar duas datas finais")
    
    datas_finais = datas[-2:]
    return [trimestre_from_date(d) for d in datas_finais]

['01/01/2025', '31/12/2024', '31/12/2025', '31/12/2024', '31/12/2025', '31/12/2024', '31/12/2025', '31/12/2024', '31/12/2025']


In [7]:
extract_anos(trecho)

['4T24', '4T25']

In [8]:
# função para reconhecer pontos e virgulas
def conv(x: str):
    x = x.strip()

    if x == "—":
        return None

    # valor contábil negativo (inteiro ou com milhar)
    if re.fullmatch(r"\(\d{1,3}(?:\.\d{3})*\)", x):
        x = "-" + x[1:-1]

    x = x.replace(".", "")
    return float(x)

In [9]:

def parse_linha_estagio(linha):
    partes = linha.split()
    estagio = int(partes[1])

    numeros = partes[2:]
    numeros = [conv(x) for x in numeros]

    return {
        "estagio": estagio,
        "exp_bruta_atual": numeros[0] if len(numeros) > 0 else None,
        "pe_atual": numeros[1] if len(numeros) > 1 else None,
        "exp_bruta_anterior": numeros[2] if len(numeros) > 2 else None,
        "pe_anterior": numeros[3] if len(numeros) > 3 else None,
    }


In [10]:
# separar em blocos para facilitar a visualização:

# tentar resolver o problema sozinho

padrao = re.compile(
    r"(Estágio1.*?)"
    r"(Estágio2.*?)"
    r"(Estágio3.*)",
    re.S | re.M
)

blocos = {"Estágio1": "", "Estágio2": "", "Estágio3": ""}

for m in padrao.finditer(texto_reduzido):
    for i, estagio in enumerate(["Estágio1", "Estágio2", "Estágio3"], start=1):
        if m.group(i):
            blocos[estagio] = m.group(i).strip()

# resultado
for k, v in blocos.items():
    print(f"\n--- {k} ---\n{v}")


--- Estágio1 ---
Estágio1
PessoasFísicas 347.749 410.807
PessoasJurídicas 332.440 359.265

--- Estágio2 ---
Estágio2
PessoasFísicas 66.468 34.869
PessoasJurídicas 13.237 9.746

--- Estágio3 ---
Estágio3
PessoasFísicas 31.357 27.550
PessoasJurídicas 11.956 11.277


In [11]:
def parse_linha_produto(linha: str):

    partes = linha.split()
    produto_raw = partes[0]
    valores = partes[1:]

    valores = [conv(v) for v in valores]

    return produto_raw, valores


In [12]:
anos = extract_anos(trecho)  # ex: ['4T24', '3T25']

MAP_PRODUTO = {
    "PessoasFísicas": "PF",
    "PessoasJurídicas": "PJ"
}

linhas_df = []

for estagio_nome, bloco in blocos.items():
    estagio_num = int(estagio_nome.replace("Estágio", ""))

    for linha in bloco.splitlines():
        if eh_produto(linha):
            produto_raw, valores = parse_linha_produto(linha)
            produto = MAP_PRODUTO[produto_raw]

            for idx, ano in enumerate(anos):
                linhas_df.append({
                    "ano": ano,
                    "banco": "Itaú",
                    "produto": produto,
                    "estagio": estagio_num,
                    "Perda Esperada": valores[idx] if idx < len(valores) else None,
                })

df_final = pd.DataFrame(linhas_df)

df_final


,ano,banco,produto,estagio,Perda Esperada
0,4T24,Itaú,PF,1,347749.0
1,4T25,Itaú,PF,1,410807.0
2,4T24,Itaú,PJ,1,332440.0
3,4T25,Itaú,PJ,1,359265.0
4,4T24,Itaú,PF,2,66468.0
5,4T25,Itaú,PF,2,34869.0
6,4T24,Itaú,PJ,2,13237.0
7,4T25,Itaú,PJ,2,9746.0
8,4T24,Itaú,PF,3,31357.0
9,4T25,Itaú,PF,3,27550.0
